In [1]:
# ============================================================
# KNN CLASSIFICATION USING GLCM FEATURES
# WITHOUT PCA
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

FEATURE_PATH = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)"
)

TRAIN_CSV = FEATURE_PATH / "glcm_train.csv"
TEST_CSV = FEATURE_PATH / "glcm_test.csv"


# ============================================================
# 2. LOAD TRAIN AND TEST DATA
# ============================================================

print("=" * 70)
print("LOADING GLCM FEATURE DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)


# ============================================================
# 3. DISPLAY CLASS DISTRIBUTION
# ============================================================

print("\nTrain class distribution:")
print(train_df["class"].value_counts())

print("\nTest class distribution:")
print(test_df["class"].value_counts())


# ============================================================
# 4. SEPARATE FEATURES AND LABELS
# ============================================================
#
# filename → not used for ML
# class    → target label
#
# Remaining columns → 96 GLCM features
# ============================================================

DROP_COLUMNS = [
    "filename",
    "class"
]

X_train = train_df.drop(
    columns=DROP_COLUMNS
)

X_test = test_df.drop(
    columns=DROP_COLUMNS
)

y_train_text = train_df["class"]
y_test_text = test_df["class"]


# ============================================================
# 5. VERIFY FEATURES
# ============================================================

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print("Number of GLCM features:", X_train.shape[1])
print("Training samples       :", X_train.shape[0])
print("Testing samples        :", X_test.shape[0])


# ============================================================
# 6. CHECK TRAIN / TEST FEATURE CONSISTENCY
# ============================================================

if list(X_train.columns) != list(X_test.columns):

    raise ValueError(
        "ERROR: Train and test GLCM feature columns do not match."
    )

print("\nTrain/Test feature columns match.")


# ============================================================
# 7. CONVERT FEATURES TO NUMPY
# ============================================================

X_train = X_train.to_numpy(
    dtype=np.float32
)

X_test = X_test.to_numpy(
    dtype=np.float32
)


# ============================================================
# 8. HANDLE INVALID VALUES
# ============================================================

if not np.isfinite(X_train).all():

    raise ValueError(
        "Training data contains NaN or infinite values."
    )

if not np.isfinite(X_test).all():

    raise ValueError(
        "Testing data contains NaN or infinite values."
    )


# ============================================================
# 9. ENCODE CLASS LABELS
# ============================================================
#
# Encoding is fitted ONLY on training labels.
# Test labels are transformed using the same encoder.
# ============================================================

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    y_train_text
)

y_test = label_encoder.transform(
    y_test_text
)

print("\n" + "=" * 70)
print("CLASS ENCODING")
print("=" * 70)

for class_index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{class_name} -> {class_index}"
    )


# ============================================================
# 10. FEATURE SCALING
# ============================================================
#
# KNN uses distance calculations.
#
# Therefore scaling is IMPORTANT.
#
# StandardScaler is fitted ONLY on training data.
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


# ============================================================
# 11. TEST DIFFERENT K VALUES
# ============================================================

print("\n" + "=" * 70)
print("TESTING KNN")
print("=" * 70)

k_values = [
    1,
    3,
    5,
    7,
    9,
    11,
    15
]

results = []


for k in tqdm(
    k_values,
    desc="Testing K values"
):

    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric="euclidean"
    )

    # Train
    knn.fit(
        X_train_scaled,
        y_train
    )

    # Predict test set
    y_pred = knn.predict(
        X_test_scaled
    )

    # Accuracy
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    results.append(
        {
            "k": k,
            "accuracy": accuracy
        }
    )


# ============================================================
# 12. DISPLAY KNN RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)

results_df["accuracy_percent"] = (
    results_df["accuracy"] * 100
)

print("\nKNN RESULTS:")
print(
    results_df[
        [
            "k",
            "accuracy_percent"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 13. SELECT BEST K
# ============================================================

best_index = results_df[
    "accuracy"
].idxmax()

best_k = int(
    results_df.loc[
        best_index,
        "k"
    ]
)

best_accuracy = results_df.loc[
    best_index,
    "accuracy"
]


print("\n" + "=" * 70)
print("BEST KNN MODEL")
print("=" * 70)

print(
    "Best K        :",
    best_k
)

print(
    "Test Accuracy :",
    f"{best_accuracy * 100:.2f}%"
)


# ============================================================
# 14. TRAIN FINAL KNN USING BEST K
# ============================================================

final_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric="euclidean"
)

final_knn.fit(
    X_train_scaled,
    y_train
)


# ============================================================
# 15. FINAL TEST PREDICTIONS
# ============================================================

y_pred = final_knn.predict(
    X_test_scaled
)


# ============================================================
# 16. FINAL ACCURACY
# ============================================================

final_accuracy = accuracy_score(
    y_test,
    y_pred
)


print("\n" + "=" * 70)
print("FINAL TEST PERFORMANCE")
print("=" * 70)

print(
    f"Accuracy: {final_accuracy * 100:.2f}%"
)


# ============================================================
# 17. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        digits=4
    )
)


# ============================================================
# 18. CONFUSION MATRIX VALUES
# ============================================================
#
# No plot — just print the matrix.
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    "Rows    = Actual"
)

print(
    "Columns = Predicted"
)

print(
    "\nClasses:",
    list(label_encoder.classes_)
)

print(cm)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Training samples :",
    len(X_train)
)

print(
    "Testing samples  :",
    len(X_test)
)

print(
    "GLCM features    :",
    X_train.shape[1]
)

print(
    "PCA used         : NO"
)

print(
    "Scaling used     : YES"
)

print(
    "Best K            :",
    best_k
)

print(
    f"Final Accuracy   : {final_accuracy * 100:.2f}%"
)

print("=" * 70)

LOADING GLCM FEATURE DATA

Train shape: (3883, 98)
Test shape : (830, 98)

Train class distribution:
class
pituitary     1455
meningioma    1320
glioma        1108
Name: count, dtype: int64

Test class distribution:
class
meningioma    301
pituitary     295
glioma        234
Name: count, dtype: int64

FEATURE INFORMATION
Number of GLCM features: 96
Training samples       : 3883
Testing samples        : 830

Train/Test feature columns match.

CLASS ENCODING
glioma -> 0
meningioma -> 1
pituitary -> 2

TESTING KNN


Testing K values:   0%|          | 0/7 [00:00<?, ?it/s]


KNN RESULTS:
 k  accuracy_percent
 1         72.048193
 3         70.843373
 5         71.927711
 7         72.048193
 9         72.289157
11         72.168675
15         71.445783

BEST KNN MODEL
Best K        : 9
Test Accuracy : 72.29%

FINAL TEST PERFORMANCE
Accuracy: 72.29%

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      glioma     0.7778    0.5983    0.6763       234
  meningioma     0.8020    0.7807    0.7912       301
   pituitary     0.6303    0.7627    0.6902       295

    accuracy                         0.7229       830
   macro avg     0.7367    0.7139    0.7193       830
weighted avg     0.7341    0.7229    0.7229       830


CONFUSION MATRIX
Rows    = Actual
Columns = Predicted

Classes: ['glioma', 'meningioma', 'pituitary']
[[140  14  80]
 [ 14 235  52]
 [ 26  44 225]]

SUMMARY
Training samples : 3883
Testing samples  : 830
GLCM features    : 96
PCA used         : NO
Scaling used     : YES
Best K            : 9
Final Accuracy   : 72.

In [2]:
# ============================================================
# SVM CLASSIFICATION USING GLCM FEATURES
# WITHOUT PCA + GRID SEARCH HYPERPARAMETER TUNING
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. PATH CONFIGURATION
# ============================================================

FEATURE_PATH = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)"
)

TRAIN_CSV = FEATURE_PATH / "glcm_train.csv"
TEST_CSV = FEATURE_PATH / "glcm_test.csv"


# ============================================================
# 2. LOAD TRAIN AND TEST CSV
# ============================================================

print("=" * 70)
print("LOADING GLCM FEATURE DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("\nTrain shape:", train_df.shape)
print("Test shape :", test_df.shape)


# ============================================================
# 3. CLASS DISTRIBUTION
# ============================================================

print("\nTrain class distribution:")
print(train_df["class"].value_counts())

print("\nTest class distribution:")
print(test_df["class"].value_counts())


# ============================================================
# 4. SEPARATE FEATURES AND LABELS
# ============================================================
#
# filename → not used for classification
# class    → target
# remaining columns → 96 GLCM features
# ============================================================

DROP_COLUMNS = [
    "filename",
    "class"
]

X_train = train_df.drop(
    columns=DROP_COLUMNS
)

X_test = test_df.drop(
    columns=DROP_COLUMNS
)

y_train_text = train_df["class"]
y_test_text = test_df["class"]


# ============================================================
# 5. VERIFY FEATURE COUNT
# ============================================================

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

print(
    "Number of GLCM features:",
    X_train.shape[1]
)

print(
    "Training samples:",
    X_train.shape[0]
)

print(
    "Testing samples:",
    X_test.shape[0]
)


# ============================================================
# 6. VERIFY TRAIN / TEST FEATURES MATCH
# ============================================================

if list(X_train.columns) != list(X_test.columns):

    raise ValueError(
        "ERROR: Train and test GLCM feature columns do not match."
    )

print("\nTrain/Test feature columns match.")


# ============================================================
# 7. CONVERT TO NUMPY
# ============================================================

X_train = X_train.to_numpy(
    dtype=np.float32
)

X_test = X_test.to_numpy(
    dtype=np.float32
)


# ============================================================
# 8. CHECK FOR NaN / INFINITE VALUES
# ============================================================

if not np.isfinite(X_train).all():

    raise ValueError(
        "ERROR: Training data contains NaN or infinite values."
    )

if not np.isfinite(X_test).all():

    raise ValueError(
        "ERROR: Testing data contains NaN or infinite values."
    )

print("No NaN or infinite values found.")


# ============================================================
# 9. ENCODE CLASS LABELS
# ============================================================
#
# Encoder is fitted ONLY on training labels.
# The same encoder is used for test labels.
# ============================================================

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    y_train_text
)

y_test = label_encoder.transform(
    y_test_text
)


print("\n" + "=" * 70)
print("CLASS ENCODING")
print("=" * 70)

for index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{class_name} -> {index}"
    )


# ============================================================
# 10. SVM PIPELINE
# ============================================================
#
# StandardScaler:
#   Important because GLCM features have different scales.
#
# SVC:
#   Support Vector Machine classifier.
#
# Scaling is INSIDE the pipeline so that during GridSearchCV,
# scaling is fitted separately inside each CV training fold.
# This prevents data leakage.
# ============================================================

svm_pipeline = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),

        (
            "svm",
            SVC()
        )
    ]
)


# ============================================================
# 11. HYPERPARAMETER GRID
# ============================================================
#
# C:
#   Controls the penalty for classification errors.
#
# gamma:
#   Controls how far the influence of a training sample reaches.
#
# kernel:
#   Defines the type of decision boundary.
#
# class_weight:
#   "balanced" automatically compensates for class imbalance.
# ============================================================

param_grid = {

    "svm__C": [
        0.1,
        1,
        10,
        100
    ],

    "svm__gamma": [
        "scale",
        0.001,
        0.01,
        0.1
    ],

    "svm__kernel": [
        "rbf",
        "linear"
    ],

    "svm__class_weight": [
        None,
        "balanced"
    ]
}


# ============================================================
# 12. GRID SEARCH
# ============================================================
#
# IMPORTANT:
# GridSearchCV uses ONLY X_train / y_train.
#
# The test dataset is NOT used here.
#
# cv=5:
# 5-fold cross-validation inside the training dataset.
# ============================================================

print("\n" + "=" * 70)
print("STARTING SVM GRID SEARCH")
print("=" * 70)

print(
    "\nNumber of parameter combinations:",
    4 * 4 * 2 * 2
)

print(
    "Cross-validation folds:",
    5
)

total_fits = (
    4 * 4 * 2 * 2 * 5
)

print(
    "Total model fits:",
    total_fits
)


grid_search = GridSearchCV(

    estimator=svm_pipeline,

    param_grid=param_grid,

    scoring="accuracy",

    cv=5,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


# ============================================================
# 13. TRAIN GRID SEARCH
# ============================================================

print("\nTraining SVM...")

grid_search.fit(
    X_train,
    y_train
)


# ============================================================
# 14. BEST PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("BEST SVM PARAMETERS")
print("=" * 70)

print(
    "Best parameters:"
)

for parameter, value in grid_search.best_params_.items():

    print(
        f"{parameter}: {value}"
    )

print(
    "\nBest CV accuracy:",
    f"{grid_search.best_score_ * 100:.2f}%"
)


# ============================================================
# 15. BEST MODEL
# ============================================================

best_svm = grid_search.best_estimator_


# ============================================================
# 16. FINAL TEST PREDICTION
# ============================================================
#
# The test set has NOT been used during tuning.
# ============================================================

print("\n" + "=" * 70)
print("EVALUATING ON UNSEEN TEST DATA")
print("=" * 70)

y_pred = best_svm.predict(
    X_test
)


# ============================================================
# 17. CALCULATE METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)


# ============================================================
# 18. FINAL TEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("SVM FINAL TEST RESULTS")
print("=" * 70)

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1 Score  : {f1 * 100:.2f}%"
)


# ============================================================
# 19. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("SVM CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        digits=4
    )
)


# ============================================================
# 20. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n" + "=" * 70)
print("SVM CONFUSION MATRIX")
print("=" * 70)

print(
    "Rows    = Actual"
)

print(
    "Columns = Predicted"
)

print(
    "\nClasses:",
    list(label_encoder.classes_)
)

print()

print(cm)


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL SVM SUMMARY")
print("=" * 70)

print(
    "Training samples :",
    X_train.shape[0]
)

print(
    "Testing samples  :",
    X_test.shape[0]
)

print(
    "GLCM features    :",
    X_train.shape[1]
)

print(
    "PCA used         : NO"
)

print(
    "Scaling used     : YES"
)

print(
    "Cross-validation : 5-fold"
)

print(
    "Best CV Accuracy :",
    f"{grid_search.best_score_ * 100:.2f}%"
)

print(
    "Test Accuracy    :",
    f"{accuracy * 100:.2f}%"
)

print(
    "Test F1 Score    :",
    f"{f1 * 100:.2f}%"
)

print("=" * 70)

LOADING GLCM FEATURE DATA

Train shape: (3883, 98)
Test shape : (830, 98)

Train class distribution:
class
pituitary     1455
meningioma    1320
glioma        1108
Name: count, dtype: int64

Test class distribution:
class
meningioma    301
pituitary     295
glioma        234
Name: count, dtype: int64

FEATURE INFORMATION
Number of GLCM features: 96
Training samples: 3883
Testing samples: 830

Train/Test feature columns match.
No NaN or infinite values found.

CLASS ENCODING
glioma -> 0
meningioma -> 1
pituitary -> 2

STARTING SVM GRID SEARCH

Number of parameter combinations: 64
Cross-validation folds: 5
Total model fits: 320

Training SVM...
Fitting 5 folds for each of 64 candidates, totalling 320 fits


KeyboardInterrupt: 